# Demo C: Prefill vs Decode

**Workshop Part 1, Group 2** | LLM Inference at Scale

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/demos/demo_c_prefill_vs_decode.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/demos/demo_c_prefill_vs_decode.ipynb)
[![Workshop Link](https://img.shields.io/badge/Workshop-link-green)](https://molab.marimo.io/notebooks/nb_zwh3Mx9ws2JextL6FGMxUp)

**Goal:** Prove that prefill is fast (parallel) and decode is slow (sequential, memory-bound).
Show the bandwidth wall with real timing measurements.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers', 'torch', 'matplotlib', 'accelerate'])

import torch, time
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'mistralai/Mistral-7B-v0.1'
DEVICE = 'cuda'
DTYPE = torch.float16

gpu_name = torch.cuda.get_device_name(0)
gpu_bw_tbs = torch.cuda.get_device_properties(0).total_memory / 1e9  # approx
print(f'GPU: {gpu_name}')

## Load Model

In [ ]:
print('Loading Mistral-7B...')
prefill_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, device_map='auto', token=False)
prefill_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=False)
print('Loaded.')

## Experiment 1: Prefill Time vs Prompt Length

Prefill processes ALL input tokens in ONE forward pass (parallel).
It should scale roughly linearly with token count because GPU parallelizes the work.

In [ ]:
# Measure prefill time (time to process input, no generation)
PREFILL_LENGTHS = [128, 512, 1024, 2048, 4096, 8192]
BASE = 'The quick brown fox jumps over the lazy dog. '

# Warmup at max length
warmup_prefill = prefill_tokenizer(BASE * 1000, return_tensors='pt',
                                   truncation=True, max_length=PREFILL_LENGTHS[-1]).to(DEVICE)
with torch.no_grad():
    prefill_model(**warmup_prefill)
del warmup_prefill
torch.cuda.empty_cache()

prefill_times = []  # (n_tokens, ms)
for plen in PREFILL_LENGTHS:
    prefill_inp = prefill_tokenizer(BASE * (plen // 10), return_tensors='pt',
                                    truncation=True, max_length=plen).to(DEVICE)
    prefill_n = prefill_inp['input_ids'].shape[1]

    torch.cuda.synchronize()
    prefill_t0 = time.perf_counter()
    with torch.no_grad():
        prefill_model(**prefill_inp)  # Just forward pass, no generation
    torch.cuda.synchronize()
    prefill_ms = (time.perf_counter() - prefill_t0) * 1000

    prefill_times.append((prefill_n, prefill_ms))
    print(f'  Prefill {prefill_n:>5} tokens: {prefill_ms:>7.1f} ms')
    del prefill_inp

print(f'\nPrefill scales linearly. GPU parallelizes the matrix multiply.')

## Experiment 2: Decode Time Per Token

Decode generates ONE token per step. Each step reads ALL model weights from HBM.
Let's measure how long each decode step takes.

In [ ]:
# Measure per-token decode latency
DECODE_PROMPT = 'Explain the theory of relativity in detail:'
N_DECODE_TOKENS = 30

decode_inp = prefill_tokenizer(DECODE_PROMPT, return_tensors='pt').to(DEVICE)

# Warmup
with torch.no_grad():
    prefill_model.generate(**decode_inp, max_new_tokens=1, pad_token_id=prefill_tokenizer.eos_token_id)

# Generate token by token, timing each step
decode_times_ms = []
decode_ids = decode_inp['input_ids'].clone()

for decode_step in range(N_DECODE_TOKENS):
    torch.cuda.synchronize()
    decode_t0 = time.perf_counter()
    with torch.no_grad():
        decode_out = prefill_model(decode_ids)
        decode_next = decode_out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    torch.cuda.synchronize()
    decode_step_ms = (time.perf_counter() - decode_t0) * 1000
    decode_times_ms.append(decode_step_ms)
    decode_ids = torch.cat([decode_ids, decode_next], dim=1)

decode_avg_ms = sum(decode_times_ms) / len(decode_times_ms)
decode_tok_per_s = 1000 / decode_avg_ms

print(f'Average decode latency: {decode_avg_ms:.1f} ms/token')
print(f'Decode throughput (1 user): {decode_tok_per_s:.0f} tok/s')
print(f'\nEach step reads 14.5 GB from HBM. At 2 TB/s that is ~7 ms minimum.')
print(f'Measured: {decode_avg_ms:.1f} ms. The rest is compute + overhead.')

## Visualization: Prefill vs Decode

In [ ]:
# Side by side: prefill time vs decode time
fig_pd, (ax_prefill, ax_decode) = plt.subplots(1, 2, figsize=(12, 4))

# Prefill chart
prefill_n_list = [r[0] for r in prefill_times]
prefill_ms_list = [r[1] for r in prefill_times]
ax_prefill.bar(range(len(prefill_n_list)), prefill_ms_list,
               color='#dcfce7', edgecolor='#000', linewidth=1.2)
ax_prefill.set_xticks(range(len(prefill_n_list)))
ax_prefill.set_xticklabels([str(n) for n in prefill_n_list])
ax_prefill.set_xlabel('Input Tokens')
ax_prefill.set_ylabel('Time (ms)')
ax_prefill.set_title('Prefill: Scales with Input (Parallel)', fontweight='bold')
for pi, pv in enumerate(prefill_ms_list):
    ax_prefill.text(pi, pv + max(prefill_ms_list)*0.02, f'{pv:.0f}', ha='center', fontsize=9)
ax_prefill.spines['top'].set_visible(False)
ax_prefill.spines['right'].set_visible(False)

# Decode chart
ax_decode.plot(range(1, N_DECODE_TOKENS+1), decode_times_ms,
              color='#991b1b', linewidth=1.5, marker='o', markersize=3)
ax_decode.axhline(y=decode_avg_ms, color='#64748b', linestyle='--',
                  label=f'Avg: {decode_avg_ms:.1f} ms/tok')
ax_decode.set_xlabel('Decode Step')
ax_decode.set_ylabel('Time (ms)')
ax_decode.set_title('Decode: ~Constant Per Token (Memory-Bound)', fontweight='bold')
ax_decode.legend()
ax_decode.spines['top'].set_visible(False)
ax_decode.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f'\nLeft: Prefill processes 8K tokens in ~{prefill_ms_list[-1]:.0f}ms (fast, parallel).')
print(f'Right: Each decode step takes ~{decode_avg_ms:.0f}ms (slow, sequential, reads 14.5 GB).')
print(f'\nThis is why decode is THE bottleneck. GPU compute sits 80% idle.')

## The Roofline: Why Decode Can't Use Compute

The roofline model shows where decode sits relative to the GPU's peak compute.
If you're left of the ridgeline, you're memory-bound. No amount of faster math helps.

In [ ]:
# Roofline plot: shows decode is 80x below compute peak
fig_roof, ax_roof = plt.subplots(figsize=(9, 5))

# GPU specs (A100)
peak_flops_tflops = 312  # FP16 TFLOPS
hbm_bw_tbs = 2.0  # TB/s

# Arithmetic intensity range
import numpy as np
ai_range = np.logspace(-1, 3, 200)  # FLOP/byte

# Roofline: min(peak_compute, bandwidth * AI)
roofline_vals = np.minimum(peak_flops_tflops, hbm_bw_tbs * 1000 * ai_range / 1e12 * 1e12)
# Simplified: performance = min(peak, bw * AI)
roof_perf = np.minimum(peak_flops_tflops, hbm_bw_tbs * ai_range)  # TFLOPS

ax_roof.loglog(ai_range, roof_perf, 'k-', linewidth=2, label='Roofline (A100)')

# Mark decode point
decode_ai = 2  # FLOP/byte for decode
decode_perf = min(peak_flops_tflops, hbm_bw_tbs * decode_ai)  # TFLOPS achieved
ax_roof.plot(decode_ai, decode_perf, 'ro', markersize=14, zorder=5, label=f'Decode (AI={decode_ai})')
ax_roof.annotate(f'Decode: {decode_ai} FLOP/byte\n{decode_perf:.0f} TFLOPS used\n(of {peak_flops_tflops} available)',
                 xy=(decode_ai, decode_perf), xytext=(20, decode_perf * 5),
                 fontsize=10, color='#991b1b', fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color='#991b1b'))

# Mark ridgeline
ridgeline_ai = peak_flops_tflops / hbm_bw_tbs  # 156 FLOP/byte
ax_roof.axvline(x=ridgeline_ai, color='#64748b', linestyle='--', linewidth=1,
                label=f'Ridgeline: {ridgeline_ai:.0f} FLOP/byte')

ax_roof.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=11)
ax_roof.set_ylabel('Performance (TFLOPS)', fontsize=11)
ax_roof.set_title('Roofline Model: Decode is 80x Below Compute Peak', fontsize=12, fontweight='bold')
ax_roof.legend(fontsize=10)
ax_roof.set_xlim(0.1, 1000)
ax_roof.set_ylim(0.1, 500)
ax_roof.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nDecode arithmetic intensity: {decode_ai} FLOP/byte')
print(f'Ridgeline (where compute becomes bottleneck): {ridgeline_ai:.0f} FLOP/byte')
print(f'Gap: {ridgeline_ai/decode_ai:.0f}x — decode uses {decode_perf/peak_flops_tflops*100:.1f}% of available compute')


## Summary

| Phase | What happens | Speed | Bottleneck |
|-------|-------------|-------|------------|
| Prefill | All tokens in parallel | Fast | Compute |
| Decode | One token at a time, reads all weights | Slow | Memory bandwidth |

**Key insight:** Decode reads 14.5 GB from HBM every step. At 2 TB/s, that's 7+ ms minimum.
No amount of faster compute fixes this. You need to read LESS or batch MORE.